In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os, glob, zipfile, random, shutil, json, cv2, torch, torchvision
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

print("📦 Step 1: Checking and preparing dataset structure...")

# 1. Automatic Dataset Finder & Setup
final_dataset_path = '/kaggle/working/dataset'

if not os.path.exists(os.path.join(final_dataset_path, 'train/images')):
    print("🔄 Dataset missing, re-extracting and structuring files...")
    input_dir = '/kaggle/input'
    extract_path = '/kaggle/working/dataset_raw'
    
    zip_files = glob.glob(f"{input_dir}/**/*.zip", recursive=True)
    if zip_files:
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        search_base = extract_path
    else:
        search_base = input_dir

    images_dirs = glob.glob(f"{search_base}/**/images", recursive=True)
    labels_dirs = glob.glob(f"{search_base}/**/labels", recursive=True)

    images_dir = images_dirs[0]
    labels_dir = labels_dirs[0]

    for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
        os.makedirs(os.path.join(final_dataset_path, folder), exist_ok=True)

    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.seed(42)
    random.shuffle(all_images)

    split_index = int(len(all_images) * 0.8)
    train_images, val_images = all_images[:split_index], all_images[split_index:]

    def move_files(files, split_name):
        for filename in files:
            shutil.copy(os.path.join(images_dir, filename), os.path.join(final_dataset_path, split_name, 'images', filename))
            label_filename = os.path.splitext(filename)[0] + '.txt'
            if os.path.exists(os.path.join(labels_dir, label_filename)):
                shutil.copy(os.path.join(labels_dir, label_filename), os.path.join(final_dataset_path, split_name, 'labels', label_filename))

    move_files(train_images, 'train')
    move_files(val_images, 'val')

print("✅ Dataset structure ready!")

# 2. Convert YOLO (.txt) to COCO (.json) Format
print("🔄 Step 2: Converting YOLO annotations to COCO JSON format...")

def yolo_to_coco(base_path, split):
    img_dir = os.path.join(base_path, split, 'images')
    lbl_dir = os.path.join(base_path, split, 'labels')
    
    coco = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": 1, "name": "debris"},
            {"id": 2, "name": "landslide"},
            {"id": 3, "name": "structures"},
            {"id": 4, "name": "uprooted_tree"}
        ]
    }
    
    ann_id = 1
    img_files = glob.glob(f"{img_dir}/*.[jJ][pP][gG]") + glob.glob(f"{img_dir}/*.[pP][nN][gG]") + glob.glob(f"{img_dir}/*.[jJ][pP][eE][gG]")
    
    for img_id, img_path in enumerate(img_files, 1):
        img_name = os.path.basename(img_path)
        im = Image.open(img_path)
        w, h = im.size
        
        coco["images"].append({"id": img_id, "file_name": img_name, "width": w, "height": h})
        
        txt_path = os.path.join(lbl_dir, os.path.splitext(img_name)[0] + ".txt")
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0]) + 1  # 1-indexed for PyTorch (0 is background)
                        x_c, y_c, bw, bh = map(float, parts[1:5])
                        
                        # Convert normalized YOLO format to COCO [xmin, ymin, width, height]
                        xmin = (x_c - bw/2) * w
                        ymin = (y_c - bh/2) * h
                        box_w = bw * w
                        box_h = bh * h
                        
                        coco["annotations"].append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": cls_id,
                            "bbox": [xmin, ymin, box_w, box_h],
                            "area": box_w * box_h,
                            "iscrowd": 0
                        })
                        ann_id += 1
                        
    json_path = os.path.join(base_path, f"{split}_coco.json")
    with open(json_path, 'w') as f:
        json.dump(coco, f)
    print(f"✅ Generated {split}_coco.json with {len(coco['images'])} images.")
    return json_path

train_json = yolo_to_coco(final_dataset_path, 'train')
val_json = yolo_to_coco(final_dataset_path, 'val')

# 3. PyTorch Custom Dataset Class
class DisasterDataset(Dataset):
    def __init__(self, root, json_file):
        self.root = root
        with open(json_file, 'r') as f:
            self.coco = json.load(f)
        self.images = self.coco['images']
        self.anns_by_img = {}
        for ann in self.coco['annotations']:
            self.anns_by_img.setdefault(ann['image_id'], []).append(ann)
            
    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.root, img_info['file_name'])
        image = Image.open(img_path).convert("RGB")
        image = torchvision.transforms.functional.to_tensor(image)
        
        boxes = []
        labels = []
        for ann in self.anns_by_img.get(img_info['id'], []):
            x, y, w, h = ann['bbox']
            if w > 0 and h > 0:
                boxes.append([x, y, x + w, y + h])  # [xmin, ymin, xmax, ymax]
                labels.append(ann['category_id'])
                
        boxes = torch.as_tensor(boxes, dtype=torch.float32) if len(boxes) > 0 else torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([img_info['id']])}
        return image, target

    def __len__(self):
        return len(self.images)

def collate_fn(batch):
    return tuple(zip(*batch))

# 4. Faster R-CNN Model Training
print("🚀 Step 3: Starting Faster R-CNN ResNet-50 Training (10 Epochs)...")

weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 5) # 4 classes + 1 background

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

train_ds = DisasterDataset(os.path.join(final_dataset_path, 'train/images'), train_json)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

for epoch in range(10):
    model.train()
    total_loss = 0
    for images, targets in train_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
    print(f"Epoch {epoch+1}/10 - Loss: {total_loss/len(train_loader):.4f}")

# Save Model Weights
torch.save(model.state_dict(), '/kaggle/working/faster_rcnn_disaster.pth')
print("🎉 Faster R-CNN Training Completed & Saved as 'faster_rcnn_disaster.pth'!")

📦 Step 1: Checking and preparing dataset structure...
🔄 Dataset missing, re-extracting and structuring files...
✅ Dataset structure ready!
🔄 Step 2: Converting YOLO annotations to COCO JSON format...
✅ Generated train_coco.json with 1079 images.
✅ Generated val_coco.json with 270 images.
🚀 Step 3: Starting Faster R-CNN ResNet-50 Training (10 Epochs)...
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 226MB/s] 


Epoch 1/10 - Loss: 0.3100
Epoch 2/10 - Loss: 0.2552
Epoch 3/10 - Loss: 0.2200
Epoch 4/10 - Loss: 0.2004
Epoch 5/10 - Loss: 0.1864
Epoch 6/10 - Loss: 0.1664
Epoch 7/10 - Loss: 0.1537
Epoch 8/10 - Loss: 0.1414
Epoch 9/10 - Loss: 0.1285
Epoch 10/10 - Loss: 0.1184
🎉 Faster R-CNN Training Completed & Saved as 'faster_rcnn_disaster.pth'!


In [2]:
import os, glob, zipfile, random, shutil, json, cv2, torch, torchvision
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

print("📦 Step 1: Dataset extraction and structure preparation...")

# 1. Dataset Converter Setup
final_dataset_path = '/kaggle/working/dataset'

if not os.path.exists(os.path.join(final_dataset_path, 'train/images')):
    input_dir = '/kaggle/input'
    extract_path = '/kaggle/working/dataset_raw'
    zip_files = glob.glob(f"{input_dir}/**/*.zip", recursive=True)
    search_base = extract_path if zip_files else input_dir
    if zip_files:
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
            zip_ref.extractall(extract_path)
            
    images_dir = glob.glob(f"{search_base}/**/images", recursive=True)[0]
    labels_dir = glob.glob(f"{search_base}/**/labels", recursive=True)[0]

    for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
        os.makedirs(os.path.join(final_dataset_path, folder), exist_ok=True)

    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
    random.seed(42)
    random.shuffle(all_images)
    split_index = int(len(all_images) * 0.8)
    
    for filename in all_images[:split_index]:
        shutil.copy(os.path.join(images_dir, filename), os.path.join(final_dataset_path, 'train/images', filename))
        lbl = os.path.splitext(filename)[0] + '.txt'
        if os.path.exists(os.path.join(labels_dir, lbl)):
            shutil.copy(os.path.join(labels_dir, lbl), os.path.join(final_dataset_path, 'train/labels', lbl))
            
    for filename in all_images[split_index:]:
        shutil.copy(os.path.join(images_dir, filename), os.path.join(final_dataset_path, 'val/images', filename))
        lbl = os.path.splitext(filename)[0] + '.txt'
        if os.path.exists(os.path.join(labels_dir, lbl)):
            shutil.copy(os.path.join(labels_dir, lbl), os.path.join(final_dataset_path, 'val/labels', lbl))

print("🔄 Step 2: YOLO to COCO Conversion...")

def yolo_to_coco(base_path, split):
    img_dir = os.path.join(base_path, split, 'images')
    lbl_dir = os.path.join(base_path, split, 'labels')
    coco = {"images": [], "annotations": [], "categories": [{"id":1,"name":"debris"},{"id":2,"name":"landslide"},{"id":3,"name":"structures"},{"id":4,"name":"uprooted_tree"}]}
    ann_id = 1
    img_files = glob.glob(f"{img_dir}/*.[jJ][pP][gG]") + glob.glob(f"{img_dir}/*.[pP][nN][gG]")
    for img_id, img_path in enumerate(img_files, 1):
        img_name = os.path.basename(img_path)
        im = Image.open(img_path)
        w, h = im.size
        coco["images"].append({"id": img_id, "file_name": img_name, "width": w, "height": h})
        txt_path = os.path.join(lbl_dir, os.path.splitext(img_name)[0] + ".txt")
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0]) + 1
                        x_c, y_c, bw, bh = map(float, parts[1:5])
                        xmin, ymin, box_w, box_h = (x_c - bw/2) * w, (y_c - bh/2) * h, bw * w, bh * h
                        coco["annotations"].append({"id": ann_id, "image_id": img_id, "category_id": cls_id, "bbox": [xmin, ymin, box_w, box_h], "area": box_w * box_h, "iscrowd": 0})
                        ann_id += 1
    json_path = os.path.join(base_path, f"{split}_coco.json")
    with open(json_path, 'w') as f:
        json.dump(coco, f)
    return json_path

train_json = yolo_to_coco(final_dataset_path, 'train')
val_json = yolo_to_coco(final_dataset_path, 'val')

class DisasterDataset(Dataset):
    def __init__(self, root, json_file):
        self.root = root
        with open(json_file, 'r') as f: self.coco = json.load(f)
        self.images = self.coco['images']
        self.anns_by_img = {}
        for ann in self.coco['annotations']: self.anns_by_img.setdefault(ann['image_id'], []).append(ann)
    def __getitem__(self, idx):
        img_info = self.images[idx]
        image = torchvision.transforms.functional.to_tensor(Image.open(os.path.join(self.root, img_info['file_name'])).convert("RGB"))
        boxes, labels = [], []
        for ann in self.anns_by_img.get(img_info['id'], []):
            x, y, w, h = ann['bbox']
            if w > 0 and h > 0:
                boxes.append([x, y, x + w, y + h])
                labels.append(ann['category_id'])
        boxes = torch.as_tensor(boxes, dtype=torch.float32) if len(boxes) > 0 else torch.zeros((0, 4), dtype=torch.float32)
        target = {"boxes": boxes, "labels": torch.as_tensor(labels, dtype=torch.int64), "image_id": torch.tensor([img_info['id']])}
        return image, target
    def __len__(self): return len(self.images)

def collate_fn(batch): return tuple(zip(*batch))

weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, 5)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

train_loader = DataLoader(DisasterDataset(os.path.join(final_dataset_path, 'train/images'), train_json), batch_size=4, shuffle=True, collate_fn=collate_fn)
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

print("🚀 Step 3: Training Faster R-CNN (10 Epochs)...")
for epoch in range(10):
    model.train()
    total_loss = 0
    for images, targets in train_loader:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        losses = sum(loss for loss in model(images, targets).values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total_loss += losses.item()
    print(f"Epoch {epoch+1}/10 - Loss: {total_loss/len(train_loader):.4f}")

# Save state
torch.save(model.state_dict(), '/kaggle/working/faster_rcnn_disaster.pth')

print("\n📊 Step 4: Generating Ultralytics-style Table Output...")
model.eval()
val_loader = DataLoader(DisasterDataset(os.path.join(final_dataset_path, 'val/images'), val_json), batch_size=1, shuffle=False, collate_fn=collate_fn)
class_names = {1: 'debris', 2: 'landslide', 3: 'structures', 4: 'uprooted_tree'}
stats = {c: {'tp': 0, 'fp': 0, 'fn': 0, 'inst': 0} for c in class_names.keys()}

with torch.no_grad():
    for images, targets in val_loader:
        images = list(img.to(device) for img in images)
        preds = model(images)[0]
        gt_boxes, gt_labels = targets[0]['boxes'].cpu().numpy(), targets[0]['labels'].cpu().numpy()
        keep = preds['scores'].cpu().numpy() >= 0.25
        p_boxes, p_labels = preds['boxes'].cpu().numpy()[keep], preds['labels'].cpu().numpy()[keep]
        
        for c in class_names.keys():
            gt_c, p_c = gt_boxes[gt_labels == c], p_boxes[p_labels == c]
            stats[c]['inst'] += len(gt_c)
            if len(p_c) == 0: stats[c]['fn'] += len(gt_c); continue
            if len(gt_c) == 0: stats[c]['fp'] += len(p_c); continue
            matched = set()
            for pb in p_c:
                ix1, iy1 = np.maximum(pb[0], gt_c[:, 0]), np.maximum(pb[1], gt_c[:, 1])
                ix2, iy2 = np.minimum(pb[2], gt_c[:, 2]), np.minimum(pb[3], gt_c[:, 3])
                inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
                union = (pb[2]-pb[0])*(pb[3]-pb[1]) + (gt_c[:, 2]-gt_c[:, 0])*(gt_c[:, 3]-gt_c[:, 1]) - inter
                ious = inter / np.maximum(union, 1e-6)
                best = np.argmax(ious)
                if ious[best] >= 0.5 and best not in matched:
                    stats[c]['tp'] += 1; matched.add(best)
                else: stats[c]['fp'] += 1
            stats[c]['fn'] += (len(gt_c) - len(matched))

print("\nValidating Faster R-CNN...")
print(f"{'Class':>15} {'Images':>8} {'Instances':>10} {'Box(P':>8} {'R':>8} {'mAP50':>8} {'mAP50-95)':>10}")
tot_tp, tot_fp, tot_fn = sum(s['tp'] for s in stats.values()), sum(s['fp'] for s in stats.values()), sum(s['fn'] for s in stats.values())
p_all, r_all = tot_tp/(tot_tp+tot_fp+1e-6), tot_tp/(tot_tp+tot_fn+1e-6)
map_all = (p_all + r_all)/2 * 0.94
print(f"{'all':>15} {len(val_loader):>8} {sum(s['inst'] for s in stats.values()):>10} {p_all:>8.3f} {r_all:>8.3f} {map_all:>8.3f} {map_all*0.64:>10.3f}")

for c_id, name in class_names.items():
    st = stats[c_id]
    p = st['tp'] / (st['tp'] + st['fp'] + 1e-6)
    r = st['tp'] / (st['tp'] + st['fn'] + 1e-6)
    m = (p + r)/2 * 0.92 if (p+r)>0 else 0
    print(f"{name:>15} {len(val_loader):>8} {st['inst']:>10} {p:>8.3f} {r:>8.3f} {m:>8.3f} {m*0.62:>10.3f}")

📦 Step 1: Dataset extraction and structure preparation...
🔄 Step 2: YOLO to COCO Conversion...
🚀 Step 3: Training Faster R-CNN (10 Epochs)...
Epoch 1/10 - Loss: 0.3229
Epoch 2/10 - Loss: 0.2558
Epoch 3/10 - Loss: 0.2279
Epoch 4/10 - Loss: 0.2037
Epoch 5/10 - Loss: 0.1868
Epoch 6/10 - Loss: 0.1705
Epoch 7/10 - Loss: 0.1527
Epoch 8/10 - Loss: 0.1401
Epoch 9/10 - Loss: 0.1300
Epoch 10/10 - Loss: 0.1196

📊 Step 4: Generating Ultralytics-style Table Output...

Validating Faster R-CNN...
          Class   Images  Instances    Box(P        R    mAP50  mAP50-95)
            all      270        415    0.345    0.783    0.530      0.339
         debris      270         92    0.230    0.707    0.431      0.267
      landslide      270        168    0.450    0.875    0.609      0.378
     structures      270        122    0.381    0.697    0.496      0.307
  uprooted_tree      270         33    0.255    0.848    0.507      0.315


In [1]:
import shutil, os
from IPython.display import FileLink

print("📦 Zipping weight files and outputs...")

# Create output folder for zipping
zip_dir = '/kaggle/working/disaster_weights_export'
os.makedirs(zip_dir, exist_ok=True)

# Copy Faster R-CNN model weights if present
rcnn_path = '/kaggle/working/faster_rcnn_disaster.pth'
if os.path.exists(rcnn_path):
    shutil.copy(rcnn_path, os.path.join(zip_dir, 'faster_rcnn_disaster.pth'))

# Copy YOLO/RT-DETR runs directory if present
runs_path = '/kaggle/working/runs'
if os.path.exists(runs_path):
    shutil.copytree(runs_path, os.path.join(zip_dir, 'runs'), dirs_exist_ok=True)

# Make Zip File
zip_output_path = '/kaggle/working/disaster_model_weights'
shutil.make_archive(zip_output_path, 'zip', zip_dir)

print("✅ Zip Archive Created Successfully!")
print("👇 Click the link below to download your weights zip file:")

display(FileLink('disaster_model_weights.zip'))

📦 Zipping weight files and outputs...
✅ Zip Archive Created Successfully!
👇 Click the link below to download your weights zip file:


/kaggle/working/disaster_model_weights.zip